In [1]:
import pandas as pd
import scipy.stats as stats
import altair as alt
import numpy as np
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportions_ztest

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [2]:
sge_set=pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260512_SGESplicingSet.xlsx')
curated_set=pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260512_CuratedSplicingSet.xlsx')

In [3]:
# Maps SGE simplified_consequence labels → curated simplified_consequence labels
CONSEQUENCE_MAP = {
    'Intron':           ['intron_variant'],
    'Splice Region':    ['splice_donor_region_variant', 'splice_polypyrimidine_tract_variant',
                         'splice_donor_5th_base_variant', 'splice_region_variant'],
    'Canonical Splice': ['splice_acceptor_variant', 'splice_donor_variant'],
    'Missense':         ['missense_variant'],
    'Synonymous':       ['synonymous_variant'],
}

SGE_LABEL_COL    = 'auth_reported_func_class'
SGE_POS_LABEL    = 'functionally_abnormal'
SGE_NEG_LABEL    = 'functionally_normal'

CURATED_LABEL_COL = 'splice_consequence'
CURATED_POS_LABEL = 'abnormal'
CURATED_NEG_LABEL = 'normal'


def build_confusion_matrix(df, label_col, pos_label, neg_label,
                            score_col='maxSpliceAI', threshold=0.2):
    """Returns [[TN, FP], [FN, TP]] at a fixed threshold."""
    sub = df[df[label_col].isin([pos_label, neg_label])].dropna(subset=[score_col])
    pos = sub[sub[label_col] == pos_label]
    neg = sub[sub[label_col] == neg_label]
    tp = int((pos[score_col] >= threshold).sum())
    fn = int((pos[score_col] <  threshold).sum())
    tn = int((neg[score_col] <  threshold).sum())
    fp = int((neg[score_col] >= threshold).sum())
    return np.array([[tn, fp], [fn, tp]])


def compare_components(cm_A, cm_B, label_A='SGE', label_B='Curated'):
    """
    Compare sensitivity and specificity between two confusion matrices.
    cm_A, cm_B: [[TN, FP], [FN, TP]]
    Returns a dict with per-metric (value_A, value_B, p_bonferroni).
    Bonferroni correction applied for two simultaneous tests.
    """
    tn_a, fp_a, fn_a, tp_a = cm_A[0,0], cm_A[0,1], cm_A[1,0], cm_A[1,1]
    tn_b, fp_b, fn_b, tp_b = cm_B[0,0], cm_B[0,1], cm_B[1,0], cm_B[1,1]

    _, p_sens = proportions_ztest([tp_a, tp_b], [tp_a + fn_a, tp_b + fn_b])
    _, p_spec = proportions_ztest([tn_a, tn_b], [tn_a + fp_a, tn_b + fp_b])

    return {
        'sensitivity': {
            label_A: tp_a / (tp_a + fn_a),
            label_B: tp_b / (tp_b + fn_b),
            'p_bonferroni': min(p_sens * 2, 1.0),
            'n_A': tp_a + fn_a,
            'n_B': tp_b + fn_b,
        },
        'specificity': {
            label_A: tn_a / (tn_a + fp_a),
            label_B: tn_b / (tn_b + fp_b),
            'p_bonferroni': min(p_spec * 2, 1.0),
            'n_A': tn_a + fp_a,
            'n_B': tn_b + fp_b,
        },
    }

In [4]:
RNA_FILTERABLE = {'Splice Region', 'Missense', 'Synonymous'}

def get_sge_sub(sge_set, group_name, rna_filtered=False):
    sub = sge_set[sge_set['simplified_consequence'] == group_name]
    if rna_filtered:
        # Restrict to variants with RNA data, then keep only RNA-confirmed LoF as positives
        sub = sub[sub['rna_consequence'].notna()]
        sub = sub[
            (sub[SGE_LABEL_COL] == SGE_NEG_LABEL) |
            ((sub[SGE_LABEL_COL] == SGE_POS_LABEL) & (sub['rna_consequence'] == 'low'))
        ]
    return sub


rows = []
for group_name, curated_consequences in CONSEQUENCE_MAP.items():
    cur_sub = curated_set[curated_set['simplified_consequence'].isin(curated_consequences)]

    variants = [(group_name, False)]
    if group_name in RNA_FILTERABLE:
        variants.append((f'{group_name} (Low RNA)', True))

    for label, rna_filtered in variants:
        sge_sub = get_sge_sub(sge_set, group_name, rna_filtered)

        sge_pos = sge_sub[SGE_LABEL_COL].eq(SGE_POS_LABEL).sum()
        sge_neg = sge_sub[SGE_LABEL_COL].eq(SGE_NEG_LABEL).sum()
        cur_pos = cur_sub[CURATED_LABEL_COL].eq(CURATED_POS_LABEL).sum()
        cur_neg = cur_sub[CURATED_LABEL_COL].eq(CURATED_NEG_LABEL).sum()

        if min(sge_pos, sge_neg, cur_pos, cur_neg) < 5:
            print(f'Skipping {label}: insufficient class counts '
                  f'(SGE +{sge_pos}/-{sge_neg}, Curated +{cur_pos}/-{cur_neg})')
            continue

        cm_sge = build_confusion_matrix(sge_sub, SGE_LABEL_COL, SGE_POS_LABEL, SGE_NEG_LABEL)
        cm_cur = build_confusion_matrix(cur_sub, CURATED_LABEL_COL, CURATED_POS_LABEL, CURATED_NEG_LABEL)

        result = compare_components(cm_sge, cm_cur)

        for metric, vals in result.items():
            rows.append({
                'consequence':  label,
                'metric':       metric,
                'SGE':          vals['SGE'],
                'Curated':      vals['Curated'],
                'delta':        vals['Curated'] - vals['SGE'],
                'p_bonferroni': vals['p_bonferroni'],
                'significant':  vals['p_bonferroni'] < 0.05,
                'n_SGE':        vals['n_A'],
                'n_Curated':    vals['n_B'],
            })

comparison_df = pd.DataFrame(rows)

fmt = lambda x: f'{x:.3f}'
sensitivity_df = comparison_df[comparison_df['metric'] == 'sensitivity'].drop(columns='metric').reset_index(drop=True)
specificity_df = comparison_df[comparison_df['metric'] == 'specificity'].drop(columns='metric').reset_index(drop=True)

print('=== Sensitivity ===')
print(sensitivity_df.to_string(index=False, float_format=fmt))
print()
print('=== Specificity ===')
print(specificity_df.to_string(index=False, float_format=fmt))

Skipping Intron: insufficient class counts (SGE +343/-8403, Curated +4/-3)
Skipping Canonical Splice: insufficient class counts (SGE +1033/-102, Curated +92/-1)
=== Sensitivity ===
            consequence   SGE  Curated  delta  p_bonferroni  significant  n_SGE  n_Curated
          Splice Region 0.747    0.873  0.126         0.035         True    747         71
Splice Region (Low RNA) 0.840    0.873  0.033         1.000        False     25         71
               Missense 0.083    0.652  0.569         0.000         True   4831         92
     Missense (Low RNA) 0.672    0.652 -0.019         1.000        False    137         92
             Synonymous 0.329    0.500  0.171         0.393        False    161         14
   Synonymous (Low RNA) 0.529    0.500 -0.029         1.000        False     17         14

=== Specificity ===
            consequence   SGE  Curated  delta  p_bonferroni  significant  n_SGE  n_Curated
          Splice Region 0.857    0.760 -0.097         0.103        Fal